# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# `to_json()` flattens the metadata to a dict, but use the object for access as per Croissant style
metadata_obj = dataset.metadata

print(f"Dataset: {getattr(metadata_obj, 'name', None)}")
print(f"Description: {getattr(metadata_obj, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets with their @id and description
record_sets = dataset.metadata.record_set
if not record_sets:
    print('No Record Sets found!')
else:
    for i, rs in enumerate(record_sets):
        print(f"[{i}] @id: {rs['@id']}")
        print(f"   Name: {rs.get('name', 'N/A')}")
        print(f"   Description: {rs.get('description', 'N/A')}")
        # List fields for each record set
        fields = rs.get('field', [])
        if fields:
            print("   Fields:")
            for field in fields:
                field_obj = field
                print(f"    - @id: {field_obj.get('@id', 'N/A')}, name: {field_obj.get('name', 'N/A')}, dataType: {field_obj.get('dataType','N/A')}")
        else:
            print("   No fields defined.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Find all record set @id values
record_sets = getattr(dataset.metadata, 'record_set', [])
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet '@id': {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for RecordSet '@id': {record_set_id}")

# Choose a specific record set for analysis (example: use the first if available)
if record_set_ids:
    chosen_record_set = record_set_ids[0]
    chosen_df = dataframes.get(chosen_record_set, pd.DataFrame())
    print(f"Sample from RecordSet {chosen_record_set}:")
    print(chosen_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Example: filter and normalize a numeric field if dataset available
if record_set_ids and not chosen_df.empty:
    # Try to select a numeric field automatically
    numeric_candidates = chosen_df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_candidates:
        print("No numeric columns found in the selected record set!")
    else:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field for analysis: {numeric_field}")
        # Remove outliers and normalize
        threshold = chosen_df[numeric_field].mean() + chosen_df[numeric_field].std()
        filtered_df = chosen_df[chosen_df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())
        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try grouping by another field
        possible_groups = [col for col in chosen_df.columns if col != numeric_field and chosen_df[col].nunique() > 1]
        if possible_groups:
            group_field = possible_groups[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(grouped_df.head())
        else:
            print("No suitable non-numeric columns for grouping found.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and not chosen_df.empty and 'numeric_field' in locals():
    # Distribution plot
    plt.figure(figsize=(8, 5))
    sns.histplot(chosen_df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    # If grouping is possible, barplot mean values
    if 'group_field' in locals():
        plt.figure(figsize=(10, 5))
        sns.barplot(
            data=chosen_df, x=group_field, y=numeric_field, ci=None
        )
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook provided an end-to-end walk-through of loading, exploring, and analyzing the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, via `mlcroissant`.

- We loaded dataset metadata and record sets using their `@id` as references for robust data access.
- Key fields and their types were discovered per record set, enabling targeted analysis.
- Data inspection and exploratory steps such as filtering, normalization, and groupwise aggregation were performed (where applicable), all referencing data by `@id`.
- Visualizations illustrated data distributions and group relationships.

Next steps could involve deeper statistical analysis, creating machine learning models, or integrating the data with additional sources for further insight into mental health trends in Kenya.